# Final Data Only Model Runner — V2 Improved

This is an edited/improved version of the earlier model runner.

It is designed for the `final_data/` or `DATA/final_fixed/` folder only. It trains stronger baselines and improved temporal models for sovereign debt distress prediction.

## Main changes from the previous runner

1. Adds stronger temporal models:
   - TCN / dilated temporal CNN
   - Attention GRU
   - Improved multimodal GRU-Transformer with separate modality encoders

2. Improves neural training:
   - Removes double class balancing from sampler + `pos_weight`
   - Uses focal loss by default for sequence models
   - Uses longer training with early stopping
   - Tracks validation AUPRC for model selection

3. Improves metric reporting:
   - AUROC
   - AUPRC
   - Brier score
   - Top 10% precision/recall/F1
   - Top 20% precision/recall/F1
   - Test metrics using validation-selected top-10% threshold

4. Improves ensembles:
   - Top neural ensemble
   - Hybrid top-model ensemble
   - Avoids blindly averaging weak models

## Recommended Colab runtime

Use **GPU runtime** for this modeling notebook:

```text
Runtime → Change runtime type → T4 GPU
```

CPU will work for classical models, but the neural sequence models will train much faster on GPU.

In [ ]:
# Install pinned dependencies. Safe to skip if already installed.
# ============================================================
# 2. Install pinned dependencies
# ============================================================

from pathlib import Path
import os
import signal

ENV_MARKER = Path('/content/.model_runner_env_ready')

if not ENV_MARKER.exists():
    !python3 -m pip install -q --upgrade --force-reinstall --no-cache-dir \
        numpy==1.26.4 \
        scipy==1.13.1 \
        pandas==2.2.2 \
        scikit-learn==1.5.2 \
        lightgbm==4.5.0 \
        xgboost==2.1.1 \
        catboost==1.2.7 \
        matplotlib==3.8.4 \
        seaborn==0.13.2 \
        PyYAML==6.0.2 \
        openpyxl

    ENV_MARKER.write_text('ready')
    print('Packages installed. Restarting runtime once so pinned versions load cleanly...')
    os.kill(os.getpid(), signal.SIGKILL)
else:
    print('Environment already prepared.')

In [ ]:
# ============================================================
# 3. Imports, config, and data loading
# ============================================================

import json
import math
import random
from pathlib import Path

import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import xgboost as xgb

from catboost import CatBoostClassifier
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, Dataset

SEED = 42
SEEDS_FOR_OPTIONAL_ENSEMBLE = [42, 77, 123]

# Core training options
RUN_OPTIONAL_SEED_ENSEMBLES = False   # Set True only if you have extra time.
LOSS_MODE_SEQ = 'focal'               # 'focal' or 'bce_pos_weight'
LOSS_MODE_FLAT_TORCH = 'bce_pos_weight'

MAX_EPOCHS_SEQ = 40
MAX_EPOCHS_FLAT_TORCH = 30
PATIENCE = 7
BATCH_SIZE_SEQ = 128
BATCH_SIZE_FLAT = 128
LR_SEQ = 5e-4
LR_FLAT_TORCH = 1e-3
WEIGHT_DECAY = 1e-4

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('DEVICE:', DEVICE)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

required = [
    'train_flat.csv', 'val_flat.csv', 'test_flat.csv',
    'train_windows.npz', 'val_windows.npz', 'test_windows.npz',
    'train_meta.csv', 'val_meta.csv', 'test_meta.csv',
    'feature_metadata.json', 'class_distribution.json'
]
missing = [name for name in required if not (FINAL_DATA_DIR / name).exists()]
if missing:
    raise FileNotFoundError(f'Missing files in final data folder: {missing}')

with open(FINAL_DATA_DIR / 'feature_metadata.json') as f:
    META = json.load(f)

FLAT_FEATURES = META['flat_features']
SEQ_FEATURES = META['feature_columns']
MODALITY_MAP = META['modality_map']
WINDOW_SIZE = META['window_size']

RESULT_DIR = FINAL_DATA_DIR / 'model_results'
PRED_DIR = RESULT_DIR / 'predictions'
PLOT_DIR = RESULT_DIR / 'plots'
MODEL_DIR = RESULT_DIR / 'saved_models'
for path in [RESULT_DIR, PRED_DIR, PLOT_DIR, MODEL_DIR]:
    path.mkdir(parents=True, exist_ok=True)

def load_flat(split):
    df = pd.read_csv(FINAL_DATA_DIR / f'{split}_flat.csv')
    X = df[FLAT_FEATURES].to_numpy(dtype=np.float32)
    y = df['distress_within_12m'].to_numpy(dtype=np.int64)
    meta = df[['iso3', 'year_month']].copy()
    meta['year_month'] = pd.to_datetime(meta['year_month']).dt.strftime('%Y-%m-%d')
    meta['y'] = y
    return X, y, meta

def load_windows(split):
    arr = np.load(FINAL_DATA_DIR / f'{split}_windows.npz')
    meta = pd.read_csv(FINAL_DATA_DIR / f'{split}_meta.csv')
    if 'window_end' in meta.columns:
        meta = meta.rename(columns={'window_end': 'year_month'})
    meta['year_month'] = pd.to_datetime(meta['year_month']).dt.strftime('%Y-%m-%d')
    if 'y' not in meta.columns:
        meta['y'] = arr['y'].astype(int)
    return arr['X'].astype(np.float32), arr['y'].astype(np.int64), meta

X_train_flat_raw, y_train_flat, meta_train_flat = load_flat('train')
X_val_flat_raw, y_val_flat, meta_val_flat = load_flat('val')
X_test_flat_raw, y_test_flat, meta_test_flat = load_flat('test')

X_train_seq_raw, y_train_seq, meta_train_seq = load_windows('train')
X_val_seq_raw, y_val_seq, meta_val_seq = load_windows('val')
X_test_seq_raw, y_test_seq, meta_test_seq = load_windows('test')

print('Flat shapes:', X_train_flat_raw.shape, X_val_flat_raw.shape, X_test_flat_raw.shape)
print('Seq shapes:', X_train_seq_raw.shape, X_val_seq_raw.shape, X_test_seq_raw.shape)
print('Train positives flat:', int(y_train_flat.sum()), '/', len(y_train_flat))
print('Train positives seq:', int(y_train_seq.sum()), '/', len(y_train_seq))
print('Modalities:', {k: len(v) for k, v in MODALITY_MAP.items()})

In [ ]:
# ============================================================
# 4. Scaling and shared inputs
# ============================================================

# Scaled versions for linear models and neural models.
flat_scaler = StandardScaler()
X_train_flat = flat_scaler.fit_transform(X_train_flat_raw).astype(np.float32)
X_val_flat = flat_scaler.transform(X_val_flat_raw).astype(np.float32)
X_test_flat = flat_scaler.transform(X_test_flat_raw).astype(np.float32)

# Tree models use raw features.
X_train_tree = X_train_flat_raw
X_val_tree = X_val_flat_raw
X_test_tree = X_test_flat_raw

# Sequence models: scale every feature using all training timesteps.
seq_scaler = StandardScaler()
n_train, t_train, f_train = X_train_seq_raw.shape
X_train_seq = seq_scaler.fit_transform(X_train_seq_raw.reshape(-1, f_train)).reshape(n_train, t_train, f_train).astype(np.float32)
X_val_seq = seq_scaler.transform(X_val_seq_raw.reshape(-1, f_train)).reshape(X_val_seq_raw.shape).astype(np.float32)
X_test_seq = seq_scaler.transform(X_test_seq_raw.reshape(-1, f_train)).reshape(X_test_seq_raw.shape).astype(np.float32)

# Flattened-window models.
X_train_flatwin_raw = X_train_seq_raw.reshape(X_train_seq_raw.shape[0], -1)
X_val_flatwin_raw = X_val_seq_raw.reshape(X_val_seq_raw.shape[0], -1)
X_test_flatwin_raw = X_test_seq_raw.reshape(X_test_seq_raw.shape[0], -1)

flatwin_scaler = StandardScaler()
X_train_flatwin = flatwin_scaler.fit_transform(X_train_flatwin_raw).astype(np.float32)
X_val_flatwin = flatwin_scaler.transform(X_val_flatwin_raw).astype(np.float32)
X_test_flatwin = flatwin_scaler.transform(X_test_flatwin_raw).astype(np.float32)

X_train_flatwin_tree = X_train_flatwin_raw.astype(np.float32)
X_val_flatwin_tree = X_val_flatwin_raw.astype(np.float32)
X_test_flatwin_tree = X_test_flatwin_raw.astype(np.float32)

scale_pos_weight_flat = max((len(y_train_flat) - y_train_flat.sum()) / max(y_train_flat.sum(), 1), 1.0)
scale_pos_weight_seq = max((len(y_train_seq) - y_train_seq.sum()) / max(y_train_seq.sum(), 1), 1.0)

print('scale_pos_weight_flat:', round(scale_pos_weight_flat, 3))
print('scale_pos_weight_seq:', round(scale_pos_weight_seq, 3))
print('Prepared scaled/raw inputs.')

In [ ]:
# ============================================================
# 5. Metrics, registration, and aligned ensembles
# ============================================================

def safe_roc_auc(y_true, y_score):
    if len(np.unique(y_true)) < 2:
        return np.nan
    return float(roc_auc_score(y_true, y_score))

def threshold_metrics(y_true, y_score, threshold):
    y_pred = (y_score >= threshold).astype(int)
    return {
        'precision': float(precision_score(y_true, y_pred, zero_division=0)),
        'recall': float(recall_score(y_true, y_pred, zero_division=0)),
        'f1': float(f1_score(y_true, y_pred, zero_division=0)),
    }

def compute_metrics(y_true, y_score):
    y_score = np.asarray(y_score, dtype=float)
    top10_threshold = float(np.quantile(y_score, 0.90))
    top20_threshold = float(np.quantile(y_score, 0.80))

    top10 = threshold_metrics(y_true, y_score, top10_threshold)
    top20 = threshold_metrics(y_true, y_score, top20_threshold)

    return {
        'auroc': safe_roc_auc(y_true, y_score),
        'auprc': float(average_precision_score(y_true, y_score)),
        'brier': float(brier_score_loss(y_true, np.clip(y_score, 0, 1))),
        'precision_at_top10pct': top10['precision'],
        'recall_at_top10pct': top10['recall'],
        'f1_at_top10pct': top10['f1'],
        'top10_threshold': top10_threshold,
        'precision_at_top20pct': top20['precision'],
        'recall_at_top20pct': top20['recall'],
        'f1_at_top20pct': top20['f1'],
        'top20_threshold': top20_threshold,
    }

def make_key(meta):
    return meta['iso3'].astype(str) + '|' + pd.to_datetime(meta['year_month']).dt.strftime('%Y-%m-%d')

def save_predictions(model_name, split, meta, scores):
    out = meta.copy()
    out['score'] = scores
    out['split'] = split
    out.to_csv(PRED_DIR / f'{model_name}__{split}.csv', index=False)

summary_rows = []
val_scores_by_model = {}
test_scores_by_model = {}
val_meta_by_model = {}
test_meta_by_model = {}

def register(model_name, val_scores, test_scores, val_meta, test_meta, val_y, test_y):
    val_scores = np.asarray(val_scores, dtype=float)
    test_scores = np.asarray(test_scores, dtype=float)

    val_scores_by_model[model_name] = val_scores
    test_scores_by_model[model_name] = test_scores
    val_meta_by_model[model_name] = val_meta.copy()
    test_meta_by_model[model_name] = test_meta.copy()

    save_predictions(model_name, 'val', val_meta, val_scores)
    save_predictions(model_name, 'test', test_meta, test_scores)

    row = {'model': model_name}

    val_m = compute_metrics(val_y, val_scores)
    val_top10_threshold = val_m['top10_threshold']

    for k, v in val_m.items():
        row[f'val__{k}'] = v

    test_m = compute_metrics(test_y, test_scores)
    for k, v in test_m.items():
        row[f'test__{k}'] = v

    # Deployment-style metric: use validation-selected top-10% threshold on test.
    test_at_val_thr = threshold_metrics(test_y, test_scores, val_top10_threshold)
    row['test__precision_at_val_top10_threshold'] = test_at_val_thr['precision']
    row['test__recall_at_val_top10_threshold'] = test_at_val_thr['recall']
    row['test__f1_at_val_top10_threshold'] = test_at_val_thr['f1']
    row['test__val_top10_threshold_used'] = val_top10_threshold

    summary_rows.append(row)
    return row

def show_result(model_name):
    df = pd.DataFrame(summary_rows)
    display(df[df['model'] == model_name].T)

def persist_summary():
    if summary_rows:
        pd.DataFrame(summary_rows).sort_values('test__auroc', ascending=False).to_csv(
            RESULT_DIR / 'summary_partial.csv',
            index=False
        )

def aligned_scores(model_name, split, ref_meta):
    if split == 'val':
        meta = val_meta_by_model[model_name]
        scores = val_scores_by_model[model_name]
    else:
        meta = test_meta_by_model[model_name]
        scores = test_scores_by_model[model_name]

    temp = meta[['iso3', 'year_month']].copy()
    temp['key'] = make_key(temp)
    temp['score'] = scores

    ref = ref_meta[['iso3', 'year_month']].copy()
    ref['key'] = make_key(ref)

    merged = ref.merge(temp[['key', 'score']], on='key', how='left')
    if merged['score'].isna().any():
        missing = merged['score'].isna().sum()
        raise ValueError(f'{model_name} missing {missing} aligned scores for {split}.')
    return merged['score'].to_numpy(dtype=float)

def rank_normalize(scores):
    s = pd.Series(scores)
    return s.rank(method='average', pct=True).to_numpy(dtype=float)

def weighted_rank_ensemble(model_names, weights, split, ref_meta):
    weights = np.asarray(weights, dtype=float)
    weights = weights / weights.sum()
    acc = np.zeros(len(ref_meta), dtype=float)
    for w, name in zip(weights, model_names):
        s = aligned_scores(name, split, ref_meta)
        acc += w * rank_normalize(s)
    return acc

print('Metric and registration utilities ready.')

In [ ]:
# ============================================================
# 6. PyTorch utilities and losses
# ============================================================

class SeqDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

class FlatDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

class FocalLossWithLogits(nn.Module):
    def __init__(self, alpha=0.75, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
    def forward(self, logits, targets):
        targets = targets.float()
        bce = nn.functional.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        pt = torch.exp(-bce)
        alpha_t = targets * self.alpha + (1 - targets) * (1 - self.alpha)
        loss = alpha_t * ((1 - pt) ** self.gamma) * bce
        return loss.mean()

def make_loss(y_train, mode='bce_pos_weight'):
    if mode == 'focal':
        return FocalLossWithLogits(alpha=0.75, gamma=2.0)
    pos = max(float(y_train.sum()), 1.0)
    neg = max(float(len(y_train) - y_train.sum()), 1.0)
    pos_weight = torch.tensor([neg / pos], dtype=torch.float32, device=DEVICE)
    return nn.BCEWithLogitsLoss(pos_weight=pos_weight)

def predict_flat_torch(model, X, batch_size=512):
    loader = DataLoader(torch.tensor(X, dtype=torch.float32), batch_size=batch_size, shuffle=False)
    preds = []
    model.eval()
    with torch.no_grad():
        for xb in loader:
            preds.append(torch.sigmoid(model(xb.to(DEVICE)).squeeze(-1)).cpu().numpy())
    return np.concatenate(preds)

def predict_seq_torch(model, X, batch_size=512):
    loader = DataLoader(torch.tensor(X, dtype=torch.float32), batch_size=batch_size, shuffle=False)
    preds = []
    model.eval()
    with torch.no_grad():
        for xb in loader:
            preds.append(torch.sigmoid(model(xb.to(DEVICE)).squeeze(-1)).cpu().numpy())
    return np.concatenate(preds)

def fit_flat_torch(
    model,
    X_train,
    y_train,
    X_val,
    y_val,
    epochs=MAX_EPOCHS_FLAT_TORCH,
    batch_size=BATCH_SIZE_FLAT,
    lr=LR_FLAT_TORCH,
    loss_mode=LOSS_MODE_FLAT_TORCH,
    patience=PATIENCE,
):
    criterion = make_loss(y_train, mode=loss_mode)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)

    train_loader = DataLoader(FlatDataset(X_train, y_train), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(FlatDataset(X_val, y_val), batch_size=batch_size, shuffle=False)

    best_state = None
    best_score = -1.0
    bad_epochs = 0

    for epoch in range(epochs):
        model.train()
        for xb, yb in train_loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)
            logits = model(xb).squeeze(-1)
            loss = criterion(logits, yb)

            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        preds, ys = [], []
        model.eval()
        with torch.no_grad():
            for xb, yb in val_loader:
                preds.append(torch.sigmoid(model(xb.to(DEVICE)).squeeze(-1)).cpu().numpy())
                ys.append(yb.numpy())
        preds = np.concatenate(preds)
        ys = np.concatenate(ys)
        score = average_precision_score(ys, preds)

        if score > best_score:
            best_score = score
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad_epochs = 0
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model

def fit_seq_torch(
    model,
    X_train,
    y_train,
    X_val,
    y_val,
    epochs=MAX_EPOCHS_SEQ,
    batch_size=BATCH_SIZE_SEQ,
    lr=LR_SEQ,
    loss_mode=LOSS_MODE_SEQ,
    patience=PATIENCE,
):
    criterion = make_loss(y_train, mode=loss_mode)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)

    train_loader = DataLoader(SeqDataset(X_train, y_train), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(SeqDataset(X_val, y_val), batch_size=batch_size, shuffle=False)

    best_state = None
    best_score = -1.0
    bad_epochs = 0

    for epoch in range(epochs):
        model.train()
        for xb, yb in train_loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)
            logits = model(xb).squeeze(-1)
            loss = criterion(logits, yb)

            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        preds, ys = [], []
        model.eval()
        with torch.no_grad():
            for xb, yb in val_loader:
                preds.append(torch.sigmoid(model(xb.to(DEVICE)).squeeze(-1)).cpu().numpy())
                ys.append(yb.numpy())
        preds = np.concatenate(preds)
        ys = np.concatenate(ys)
        score = average_precision_score(ys, preds)

        if score > best_score:
            best_score = score
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad_epochs = 0
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model

print('PyTorch utilities ready.')

In [ ]:
# ============================================================
# 7. Model classes
# ============================================================

class GRUClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim=96, num_layers=1, dropout=0.35):
        super().__init__()
        self.gru = nn.GRU(
            input_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        out, _ = self.gru(x)
        return self.head(self.dropout(out[:, -1, :]))

class AttentionGRUClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim=96, num_layers=1, dropout=0.35):
        super().__init__()
        self.gru = nn.GRU(
            input_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.attn = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.Tanh(),
            nn.Linear(hidden_dim // 2, 1)
        )
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        out, _ = self.gru(x)
        weights = torch.softmax(self.attn(out).squeeze(-1), dim=1)
        pooled = torch.sum(out * weights.unsqueeze(-1), dim=1)
        return self.head(self.dropout(pooled))

class LSTMClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim=96, num_layers=1, dropout=0.35):
        super().__init__()
        self.lstm = nn.LSTM(
            input_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(self.dropout(out[:, -1, :]))

class CNN1DClassifier(nn.Module):
    def __init__(self, input_dim, channels=128, dropout=0.35):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(input_dim, channels, kernel_size=3, padding=1),
            nn.BatchNorm1d(channels),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Conv1d(channels, channels, kernel_size=5, padding=2),
            nn.BatchNorm1d(channels),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1)
        )
        self.head = nn.Linear(channels, 1)

    def forward(self, x):
        x = x.transpose(1, 2)
        x = self.net(x).squeeze(-1)
        return self.head(x)

class Chomp1d(nn.Module):
    def __init__(self, chomp_size):
        super().__init__()
        self.chomp_size = chomp_size
    def forward(self, x):
        if self.chomp_size == 0:
            return x
        return x[:, :, :-self.chomp_size].contiguous()

class TemporalBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size, dilation, dropout):
        super().__init__()
        padding = (kernel_size - 1) * dilation
        self.conv1 = nn.Conv1d(in_ch, out_ch, kernel_size, padding=padding, dilation=dilation)
        self.chomp1 = Chomp1d(padding)
        self.bn1 = nn.BatchNorm1d(out_ch)
        self.relu1 = nn.ReLU()
        self.drop1 = nn.Dropout(dropout)

        self.conv2 = nn.Conv1d(out_ch, out_ch, kernel_size, padding=padding, dilation=dilation)
        self.chomp2 = Chomp1d(padding)
        self.bn2 = nn.BatchNorm1d(out_ch)
        self.relu2 = nn.ReLU()
        self.drop2 = nn.Dropout(dropout)

        self.downsample = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else None
        self.final_relu = nn.ReLU()

    def forward(self, x):
        out = self.conv1(x)
        out = self.chomp1(out)
        out = self.bn1(out)
        out = self.relu1(out)
        out = self.drop1(out)

        out = self.conv2(out)
        out = self.chomp2(out)
        out = self.bn2(out)
        out = self.relu2(out)
        out = self.drop2(out)

        res = x if self.downsample is None else self.downsample(x)
        return self.final_relu(out + res)

class TCNClassifier(nn.Module):
    def __init__(self, input_dim, channels=96, levels=3, kernel_size=3, dropout=0.35):
        super().__init__()
        layers = []
        in_ch = input_dim
        for i in range(levels):
            dilation = 2 ** i
            layers.append(TemporalBlock(in_ch, channels, kernel_size, dilation, dropout))
            in_ch = channels
        self.tcn = nn.Sequential(*layers)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.head = nn.Linear(channels, 1)

    def forward(self, x):
        x = x.transpose(1, 2)
        x = self.tcn(x)
        x = self.pool(x).squeeze(-1)
        return self.head(x)

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=128):
        super().__init__()
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class TransformerEncoderClassifier(nn.Module):
    def __init__(self, input_dim, d_model=96, nhead=4, num_layers=2, dropout=0.25):
        super().__init__()
        self.proj = nn.Linear(input_dim, d_model)
        self.pos = PositionalEncoding(d_model, max_len=WINDOW_SIZE + 4)
        layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=d_model * 2,
            dropout=dropout,
            batch_first=True,
            activation='gelu'
        )
        self.enc = nn.TransformerEncoder(layer, num_layers=num_layers)
        self.head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Dropout(dropout),
            nn.Linear(d_model, 1)
        )

    def forward(self, x):
        x = self.proj(x)
        x = self.pos(x)
        x = self.enc(x).mean(dim=1)
        return self.head(x)

# Modality index maps.
flat_modality_indices = {}
seq_modality_indices = {}
for name, cols in MODALITY_MAP.items():
    flat_idx = [FLAT_FEATURES.index(col) for col in cols if col in FLAT_FEATURES]
    seq_idx = [SEQ_FEATURES.index(col) for col in cols if col in SEQ_FEATURES]
    if flat_idx:
        flat_modality_indices[name] = flat_idx
    if seq_idx:
        seq_modality_indices[name] = seq_idx

print('Flat modality sizes:', {k: len(v) for k, v in flat_modality_indices.items()})
print('Seq modality sizes:', {k: len(v) for k, v in seq_modality_indices.items()})

class ImprovedMultimodalGRUTransformer(nn.Module):
    '''
    Improved version:
    - Separate projection per modality
    - Separate GRU encoder per modality
    - Learnable modality embeddings
    - Transformer fusion across modality tokens
    '''
    def __init__(self, modality_indices, d_model=96, nhead=4, num_layers=2, dropout=0.25):
        super().__init__()
        self.modality_names = list(modality_indices.keys())
        self.modality_indices = modality_indices
        self.proj = nn.ModuleDict()
        self.grus = nn.ModuleDict()

        for name, idx in modality_indices.items():
            self.proj[name] = nn.Linear(len(idx), d_model)
            self.grus[name] = nn.GRU(d_model, d_model, batch_first=True)

        self.modality_embedding = nn.Parameter(torch.randn(1, len(self.modality_names), d_model) * 0.02)

        layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=d_model * 2,
            dropout=dropout,
            batch_first=True,
            activation='gelu'
        )
        self.enc = nn.TransformerEncoder(layer, num_layers=num_layers)
        self.head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Dropout(dropout),
            nn.Linear(d_model, 1)
        )

    def forward(self, x):
        tokens = []
        for name in self.modality_names:
            idx = self.modality_indices[name]
            seq = self.proj[name](x[:, :, idx])
            _, h = self.grus[name](seq)
            tokens.append(h[-1])
        tokens = torch.stack(tokens, dim=1)
        tokens = tokens + self.modality_embedding
        fused = self.enc(tokens).mean(dim=1)
        return self.head(fused)

print('Model classes ready.')

In [ ]:
# ============================================================
# 8. Classical linear and tree baselines
# ============================================================

set_seed(SEED)

print('Training Logistic Regression...')
lr = LogisticRegression(max_iter=4000, class_weight='balanced', solver='lbfgs')
lr.fit(X_train_flat, y_train_flat)
register(
    'logistic_regression',
    lr.predict_proba(X_val_flat)[:, 1],
    lr.predict_proba(X_test_flat)[:, 1],
    meta_val_flat,
    meta_test_flat,
    y_val_flat,
    y_test_flat
)
persist_summary()
show_result('logistic_regression')

print('Training Elastic Net Logistic Regression...')
enet = LogisticRegression(
    max_iter=5000,
    class_weight='balanced',
    penalty='elasticnet',
    l1_ratio=0.35,
    solver='saga',
    C=0.5,
    n_jobs=-1,
    random_state=SEED
)
enet.fit(X_train_flat, y_train_flat)
register(
    'elastic_net_logistic_regression',
    enet.predict_proba(X_val_flat)[:, 1],
    enet.predict_proba(X_test_flat)[:, 1],
    meta_val_flat,
    meta_test_flat,
    y_val_flat,
    y_test_flat
)
persist_summary()
show_result('elastic_net_logistic_regression')

print('Training Random Forest tuned...')
rf = RandomForestClassifier(
    n_estimators=800,
    max_depth=5,
    min_samples_leaf=10,
    min_samples_split=20,
    max_features='sqrt',
    class_weight='balanced_subsample',
    n_jobs=-1,
    random_state=SEED
)
rf.fit(X_train_tree, y_train_flat)
register(
    'random_forest_tuned',
    rf.predict_proba(X_val_tree)[:, 1],
    rf.predict_proba(X_test_tree)[:, 1],
    meta_val_flat,
    meta_test_flat,
    y_val_flat,
    y_test_flat
)
persist_summary()
show_result('random_forest_tuned')

print('Training Extra Trees tuned...')
et = ExtraTreesClassifier(
    n_estimators=800,
    max_depth=5,
    min_samples_leaf=10,
    min_samples_split=20,
    max_features='sqrt',
    class_weight='balanced',
    n_jobs=-1,
    random_state=SEED
)
et.fit(X_train_tree, y_train_flat)
register(
    'extra_trees_tuned',
    et.predict_proba(X_val_tree)[:, 1],
    et.predict_proba(X_test_tree)[:, 1],
    meta_val_flat,
    meta_test_flat,
    y_val_flat,
    y_test_flat
)
persist_summary()
show_result('extra_trees_tuned')

In [ ]:
# ============================================================
# 9. Regularized boosting models
# ============================================================

set_seed(SEED)

print('Training XGBoost tuned...')
xgb_flat = xgb.XGBClassifier(
    n_estimators=2500,
    max_depth=2,
    learning_rate=0.015,
    subsample=0.70,
    colsample_bytree=0.70,
    min_child_weight=8,
    reg_lambda=12.0,
    reg_alpha=2.0,
    gamma=1.0,
    scale_pos_weight=scale_pos_weight_flat,
    eval_metric='aucpr',
    early_stopping_rounds=80,
    random_state=SEED,
    tree_method='hist',
    device='cuda' if torch.cuda.is_available() else 'cpu'
)
xgb_flat.fit(X_train_tree, y_train_flat, eval_set=[(X_val_tree, y_val_flat)], verbose=False)
register(
    'xgboost_tuned',
    xgb_flat.predict_proba(X_val_tree)[:, 1],
    xgb_flat.predict_proba(X_test_tree)[:, 1],
    meta_val_flat,
    meta_test_flat,
    y_val_flat,
    y_test_flat
)
persist_summary()
show_result('xgboost_tuned')

print('Training LightGBM tuned...')
lgbm = lgb.LGBMClassifier(
    n_estimators=2500,
    learning_rate=0.015,
    num_leaves=7,
    max_depth=3,
    min_child_samples=40,
    subsample=0.70,
    colsample_bytree=0.70,
    reg_alpha=2.0,
    reg_lambda=12.0,
    scale_pos_weight=scale_pos_weight_flat,
    random_state=SEED,
    verbosity=-1
)
lgbm.fit(
    X_train_tree,
    y_train_flat,
    eval_set=[(X_val_tree, y_val_flat)],
    eval_metric='average_precision',
    callbacks=[lgb.early_stopping(80, verbose=False)]
)
register(
    'lightgbm_tuned',
    lgbm.predict_proba(X_val_tree)[:, 1],
    lgbm.predict_proba(X_test_tree)[:, 1],
    meta_val_flat,
    meta_test_flat,
    y_val_flat,
    y_test_flat
)
persist_summary()
show_result('lightgbm_tuned')

print('Training CatBoost tuned...')
try:
    cat = CatBoostClassifier(
        iterations=2500,
        learning_rate=0.02,
        depth=3,
        l2_leaf_reg=12.0,
        loss_function='Logloss',
        eval_metric='PRAUC',
        auto_class_weights='Balanced',
        task_type='GPU' if torch.cuda.is_available() else 'CPU',
        random_seed=SEED,
        verbose=False,
        od_type='Iter',
        od_wait=100
    )
    cat.fit(X_train_tree, y_train_flat, eval_set=(X_val_tree, y_val_flat), verbose=False)
except Exception as e:
    print('CatBoost GPU failed, retrying CPU. Error:', e)
    cat = CatBoostClassifier(
        iterations=2500,
        learning_rate=0.02,
        depth=3,
        l2_leaf_reg=12.0,
        loss_function='Logloss',
        eval_metric='PRAUC',
        auto_class_weights='Balanced',
        task_type='CPU',
        random_seed=SEED,
        verbose=False,
        od_type='Iter',
        od_wait=100
    )
    cat.fit(X_train_tree, y_train_flat, eval_set=(X_val_tree, y_val_flat), verbose=False)

register(
    'catboost_tuned',
    cat.predict_proba(X_val_tree)[:, 1],
    cat.predict_proba(X_test_tree)[:, 1],
    meta_val_flat,
    meta_test_flat,
    y_val_flat,
    y_test_flat
)
persist_summary()
show_result('catboost_tuned')

In [ ]:
# ============================================================
# 10. Flattened-window XGBoost
# ============================================================

set_seed(SEED)

print('Training Flattened-window XGBoost tuned...')
xgb_flatwin = xgb.XGBClassifier(
    n_estimators=2500,
    max_depth=2,
    learning_rate=0.015,
    subsample=0.70,
    colsample_bytree=0.70,
    min_child_weight=8,
    reg_lambda=15.0,
    reg_alpha=3.0,
    gamma=1.0,
    scale_pos_weight=scale_pos_weight_seq,
    eval_metric='aucpr',
    early_stopping_rounds=80,
    random_state=SEED,
    tree_method='hist',
    device='cuda' if torch.cuda.is_available() else 'cpu'
)
xgb_flatwin.fit(X_train_flatwin_tree, y_train_seq, eval_set=[(X_val_flatwin_tree, y_val_seq)], verbose=False)

register(
    'flattened_window_xgboost_tuned',
    xgb_flatwin.predict_proba(X_val_flatwin_tree)[:, 1],
    xgb_flatwin.predict_proba(X_test_flatwin_tree)[:, 1],
    meta_val_seq,
    meta_test_seq,
    y_val_seq,
    y_test_seq
)
persist_summary()
show_result('flattened_window_xgboost_tuned')

In [ ]:
# ============================================================
# 11. Neural temporal models
# ============================================================

input_dim = X_train_seq.shape[-1]

set_seed(SEED)
print('Training CNN-1D...')
cnn = fit_seq_torch(
    CNN1DClassifier(input_dim).to(DEVICE),
    X_train_seq,
    y_train_seq,
    X_val_seq,
    y_val_seq,
    epochs=MAX_EPOCHS_SEQ,
    lr=LR_SEQ,
    loss_mode=LOSS_MODE_SEQ
)
register(
    'cnn_1d',
    predict_seq_torch(cnn, X_val_seq),
    predict_seq_torch(cnn, X_test_seq),
    meta_val_seq,
    meta_test_seq,
    y_val_seq,
    y_test_seq
)
persist_summary()
show_result('cnn_1d')

set_seed(SEED)
print('Training TCN...')
tcn = fit_seq_torch(
    TCNClassifier(input_dim).to(DEVICE),
    X_train_seq,
    y_train_seq,
    X_val_seq,
    y_val_seq,
    epochs=MAX_EPOCHS_SEQ,
    lr=LR_SEQ,
    loss_mode=LOSS_MODE_SEQ
)
register(
    'tcn',
    predict_seq_torch(tcn, X_val_seq),
    predict_seq_torch(tcn, X_test_seq),
    meta_val_seq,
    meta_test_seq,
    y_val_seq,
    y_test_seq
)
persist_summary()
show_result('tcn')

set_seed(SEED)
print('Training GRU...')
gru = fit_seq_torch(
    GRUClassifier(input_dim).to(DEVICE),
    X_train_seq,
    y_train_seq,
    X_val_seq,
    y_val_seq,
    epochs=MAX_EPOCHS_SEQ,
    lr=LR_SEQ,
    loss_mode=LOSS_MODE_SEQ
)
register(
    'gru',
    predict_seq_torch(gru, X_val_seq),
    predict_seq_torch(gru, X_test_seq),
    meta_val_seq,
    meta_test_seq,
    y_val_seq,
    y_test_seq
)
persist_summary()
show_result('gru')

set_seed(SEED)
print('Training Attention GRU...')
attn_gru = fit_seq_torch(
    AttentionGRUClassifier(input_dim).to(DEVICE),
    X_train_seq,
    y_train_seq,
    X_val_seq,
    y_val_seq,
    epochs=MAX_EPOCHS_SEQ,
    lr=LR_SEQ,
    loss_mode=LOSS_MODE_SEQ
)
register(
    'attention_gru',
    predict_seq_torch(attn_gru, X_val_seq),
    predict_seq_torch(attn_gru, X_test_seq),
    meta_val_seq,
    meta_test_seq,
    y_val_seq,
    y_test_seq
)
persist_summary()
show_result('attention_gru')

set_seed(SEED)
print('Training LSTM...')
lstm = fit_seq_torch(
    LSTMClassifier(input_dim).to(DEVICE),
    X_train_seq,
    y_train_seq,
    X_val_seq,
    y_val_seq,
    epochs=MAX_EPOCHS_SEQ,
    lr=LR_SEQ,
    loss_mode=LOSS_MODE_SEQ
)
register(
    'lstm',
    predict_seq_torch(lstm, X_val_seq),
    predict_seq_torch(lstm, X_test_seq),
    meta_val_seq,
    meta_test_seq,
    y_val_seq,
    y_test_seq
)
persist_summary()
show_result('lstm')

set_seed(SEED)
print('Training Transformer Encoder...')
trf = fit_seq_torch(
    TransformerEncoderClassifier(input_dim).to(DEVICE),
    X_train_seq,
    y_train_seq,
    X_val_seq,
    y_val_seq,
    epochs=MAX_EPOCHS_SEQ,
    lr=LR_SEQ,
    loss_mode=LOSS_MODE_SEQ
)
register(
    'transformer_encoder',
    predict_seq_torch(trf, X_val_seq),
    predict_seq_torch(trf, X_test_seq),
    meta_val_seq,
    meta_test_seq,
    y_val_seq,
    y_test_seq
)
persist_summary()
show_result('transformer_encoder')

set_seed(SEED)
print('Training Improved Multimodal GRU-Transformer...')
mm = fit_seq_torch(
    ImprovedMultimodalGRUTransformer(seq_modality_indices).to(DEVICE),
    X_train_seq,
    y_train_seq,
    X_val_seq,
    y_val_seq,
    epochs=MAX_EPOCHS_SEQ,
    lr=LR_SEQ,
    loss_mode=LOSS_MODE_SEQ
)
register(
    'improved_multimodal_gru_transformer',
    predict_seq_torch(mm, X_val_seq),
    predict_seq_torch(mm, X_test_seq),
    meta_val_seq,
    meta_test_seq,
    y_val_seq,
    y_test_seq
)
persist_summary()
show_result('improved_multimodal_gru_transformer')

In [ ]:
# ============================================================
# 12. Optional seed ensembles for top neural models
# ============================================================

def train_seeded_model(model_factory, model_prefix, seeds):
    val_preds = []
    test_preds = []

    for seed in seeds:
        set_seed(seed)
        print(f'Training {model_prefix} seed={seed}...')
        model = fit_seq_torch(
            model_factory().to(DEVICE),
            X_train_seq,
            y_train_seq,
            X_val_seq,
            y_val_seq,
            epochs=MAX_EPOCHS_SEQ,
            lr=LR_SEQ,
            loss_mode=LOSS_MODE_SEQ
        )
        val_preds.append(predict_seq_torch(model, X_val_seq))
        test_preds.append(predict_seq_torch(model, X_test_seq))

    val_ens = np.mean(np.vstack(val_preds), axis=0)
    test_ens = np.mean(np.vstack(test_preds), axis=0)

    register(
        f'{model_prefix}_seed_ensemble',
        val_ens,
        test_ens,
        meta_val_seq,
        meta_test_seq,
        y_val_seq,
        y_test_seq
    )
    persist_summary()
    show_result(f'{model_prefix}_seed_ensemble')

if RUN_OPTIONAL_SEED_ENSEMBLES:
    train_seeded_model(lambda: GRUClassifier(input_dim), 'gru', SEEDS_FOR_OPTIONAL_ENSEMBLE)
    train_seeded_model(lambda: AttentionGRUClassifier(input_dim), 'attention_gru', SEEDS_FOR_OPTIONAL_ENSEMBLE)
    train_seeded_model(lambda: ImprovedMultimodalGRUTransformer(seq_modality_indices), 'improved_multimodal_gru_transformer', SEEDS_FOR_OPTIONAL_ENSEMBLE)
else:
    print('Skipping optional seed ensembles. Set RUN_OPTIONAL_SEED_ENSEMBLES=True near the top to run them.')

In [ ]:
# ============================================================
# 13. Top-only ensembles
# ============================================================

summary_so_far = pd.DataFrame(summary_rows)

# Neural-only ensemble: avoids weak flat tree models.
neural_candidates = [
    name for name in [
        'gru',
        'attention_gru',
        'lstm',
        'tcn',
        'transformer_encoder',
        'improved_multimodal_gru_transformer',
        'gru_seed_ensemble',
        'attention_gru_seed_ensemble',
        'improved_multimodal_gru_transformer_seed_ensemble',
    ]
    if name in val_scores_by_model
]

# Hybrid ensemble: includes stable linear baselines plus strongest temporal models.
hybrid_candidates = [
    name for name in [
        'logistic_regression',
        'elastic_net_logistic_regression',
        'gru',
        'attention_gru',
        'lstm',
        'tcn',
        'improved_multimodal_gru_transformer',
        'gru_seed_ensemble',
        'attention_gru_seed_ensemble',
        'improved_multimodal_gru_transformer_seed_ensemble',
    ]
    if name in val_scores_by_model
]

def val_auprc_weight(model_names):
    df = pd.DataFrame(summary_rows)
    d = dict(zip(df['model'], df['val__auprc']))
    weights = np.array([max(float(d.get(name, 0.0)), 1e-6) for name in model_names], dtype=float)
    return weights / weights.sum()

if neural_candidates:
    w = val_auprc_weight(neural_candidates)
    val_ens = weighted_rank_ensemble(neural_candidates, w, 'val', meta_val_seq)
    test_ens = weighted_rank_ensemble(neural_candidates, w, 'test', meta_test_seq)

    register(
        'top_neural_rank_ensemble',
        val_ens,
        test_ens,
        meta_val_seq,
        meta_test_seq,
        y_val_seq,
        y_test_seq
    )

    pd.DataFrame({'model': neural_candidates, 'weight': w}).sort_values('weight', ascending=False).to_csv(
        RESULT_DIR / 'top_neural_ensemble_weights.csv',
        index=False
    )
    persist_summary()
    show_result('top_neural_rank_ensemble')

if hybrid_candidates:
    w = val_auprc_weight(hybrid_candidates)
    val_ens = weighted_rank_ensemble(hybrid_candidates, w, 'val', meta_val_flat)
    test_ens = weighted_rank_ensemble(hybrid_candidates, w, 'test', meta_test_flat)

    register(
        'hybrid_top_rank_ensemble',
        val_ens,
        test_ens,
        meta_val_flat,
        meta_test_flat,
        y_val_flat,
        y_test_flat
    )

    pd.DataFrame({'model': hybrid_candidates, 'weight': w}).sort_values('weight', ascending=False).to_csv(
        RESULT_DIR / 'hybrid_top_ensemble_weights.csv',
        index=False
    )
    persist_summary()
    show_result('hybrid_top_rank_ensemble')

In [ ]:
# ============================================================
# 14. Final summary, plots, and export
# ============================================================

summary = pd.DataFrame(summary_rows).sort_values('test__auroc', ascending=False).reset_index(drop=True)
summary.to_csv(RESULT_DIR / 'summary.csv', index=False)

print('Saved summary to:', RESULT_DIR / 'summary.csv')
display(summary)

# Compact table for report.
cols_for_report = [
    'model',
    'val__auroc',
    'val__auprc',
    'val__recall_at_top10pct',
    'test__auroc',
    'test__auprc',
    'test__precision_at_top10pct',
    'test__recall_at_top10pct',
    'test__f1_at_top10pct',
    'test__recall_at_top20pct',
    'test__recall_at_val_top10_threshold',
]
report_cols = [c for c in cols_for_report if c in summary.columns]
summary[report_cols].to_csv(RESULT_DIR / 'summary_report_table.csv', index=False)
display(summary[report_cols])

# Plot AUROC and AUPRC.
plot_df = summary[['model', 'test__auroc', 'test__auprc']].set_index('model')
ax = plot_df.plot(kind='bar', figsize=(15, 6))
ax.set_title('Test Performance by Model — V2 Improved')
ax.set_ylabel('Score')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(PLOT_DIR / 'test_model_comparison_v2.png', dpi=200)
plt.show()

# Top-10 recall plot.
recall_df = summary[['model', 'test__recall_at_top10pct', 'test__recall_at_top20pct']].set_index('model')
ax = recall_df.plot(kind='bar', figsize=(15, 6))
ax.set_title('Early Warning Recall by Risk Bucket — V2 Improved')
ax.set_ylabel('Recall')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(PLOT_DIR / 'test_recall_buckets_v2.png', dpi=200)
plt.show()

print('Result directory:', RESULT_DIR)
print('Prediction files:', PRED_DIR)
print('Plots:', PLOT_DIR)

In [ ]:
# ============================================================
# 15. Zip model results for download/sharing
# ============================================================

import shutil

zip_path = shutil.make_archive(str(FINAL_DATA_DIR / 'model_results'), 'zip', RESULT_DIR)
print('Created ZIP:', zip_path)

# Notes for interpreting V2 results

Use **test AUROC**, **test AUPRC**, and **top-risk-bucket recall** together.

For an early-warning system, the most important practical metric is often:

```text
Recall at top 10% risk
```

This answers:

```text
If we flag the riskiest 10% of country-months, how many actual distress cases do we catch?
```

The strongest final story will usually compare:

1. Best stable baseline: Logistic / Elastic Net
2. Best tree/boosting model: XGBoost, LightGBM, CatBoost, or Extra Trees
3. Best temporal model: GRU / Attention GRU / TCN / LSTM
4. Best advanced model: Improved Multimodal GRU-Transformer
5. Best ensemble: Top neural or hybrid top-model rank ensemble

# Visualizations


In [ ]:
# ============================================================
# VISUALIZATION CELLS — Add these after Cell 14 in the notebook
# Copy each block into a separate Colab cell
# ============================================================


# ════════════════════════════════════════════════════════════════
# CELL A: Setup and data preparation for all plots
# ════════════════════════════════════════════════════════════════

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
from sklearn.metrics import roc_curve, precision_recall_curve
from pathlib import Path

plt.rcParams.update({
    'figure.dpi': 150,
    'font.family': 'sans-serif',
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.titleweight': 'bold',
    'axes.labelsize': 12,
    'figure.facecolor': 'white',
})

# Country names for display
COUNTRY_NAMES = {
    'ARG':'Argentina','BRA':'Brazil','COL':'Colombia','DOM':'Dominican Rep.',
    'ECU':'Ecuador','EGY':'Egypt','ETH':'Ethiopia','GHA':'Ghana','HUN':'Hungary',
    'IDN':'Indonesia','IND':'India','JAM':'Jamaica','JOR':'Jordan','KEN':'Kenya',
    'LBN':'Lebanon','LKA':'Sri Lanka','MEX':'Mexico','MOZ':'Mozambique',
    'MYS':'Malaysia','NGA':'Nigeria','PAK':'Pakistan','PER':'Peru','PHL':'Philippines',
    'POL':'Poland','ROU':'Romania','RUS':'Russia','TUN':'Tunisia','TUR':'Turkey',
    'UKR':'Ukraine','VEN':'Venezuela','ZAF':'South Africa','ZMB':'Zambia'
}

# Models to highlight (top performers + key baselines)
HERO_MODELS = [
    'improved_multimodal_gru_transformer',
    'cnn_1d',
    'gru',
    'logistic_regression',
    'transformer_encoder',
    'lstm',
]

HERO_COLORS = {
    'improved_multimodal_gru_transformer': '#ef4444',
    'cnn_1d': '#f59e0b',
    'gru': '#22c55e',
    'logistic_regression': '#64748b',
    'transformer_encoder': '#8b5cf6',
    'lstm': '#06b6d4',
}

HERO_LABELS = {
    'improved_multimodal_gru_transformer': 'Multimodal GRU-TF',
    'cnn_1d': 'CNN-1D',
    'gru': 'GRU',
    'logistic_regression': 'Logistic Reg.',
    'transformer_encoder': 'Transformer',
    'lstm': 'LSTM',
}

# Load all predictions
def load_preds(model_name, split):
    path = PRED_DIR / f'{model_name}__{split}.csv'
    if path.exists():
        return pd.read_csv(path)
    return None

print("Visualization setup ready.")




In [ ]:

# ════════════════════════════════════════════════════════════════
# PLOT 1: Country Risk Timeline (4 crisis countries)
# "How the model's predicted risk evolved before each crisis"
# ════════════════════════════════════════════════════════════════

crisis_countries = [
    ('LKA', '2022-05-01', 'Default (May 2022)'),
    ('GHA', '2022-12-01', 'Restructuring (Dec 2022)'),
    ('ZMB', '2020-11-01', 'Default (Nov 2020)'),
    ('ETH', '2023-12-01', 'Restructuring (Dec 2023)'),
]

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

for idx, (iso3, crisis_date, crisis_label) in enumerate(crisis_countries):
    ax = axes[idx]
    crisis_dt = pd.to_datetime(crisis_date)

    for model_name in ['logistic_regression', 'gru', 'improved_multimodal_gru_transformer']:
        # Combine val + test predictions
        val_df = load_preds(model_name, 'val')
        test_df = load_preds(model_name, 'test')
        if val_df is None or test_df is None:
            continue
        combined = pd.concat([val_df, test_df]).sort_values('year_month')
        country = combined[combined['iso3'] == iso3].copy()
        if country.empty:
            continue
        country['date'] = pd.to_datetime(country['year_month'])
        ax.plot(country['date'], country['score'],
                color=HERO_COLORS[model_name],
                linewidth=2.5 if model_name == 'improved_multimodal_gru_transformer' else 1.5,
                label=HERO_LABELS[model_name],
                alpha=1.0 if model_name == 'improved_multimodal_gru_transformer' else 0.7)

    # Mark actual distress periods
    val_df = load_preds('improved_multimodal_gru_transformer', 'val')
    test_df = load_preds('improved_multimodal_gru_transformer', 'test')
    combined = pd.concat([val_df, test_df]).sort_values('year_month')
    country = combined[combined['iso3'] == iso3]
    if not country.empty:
        distress = country[country['y'] == 1]
        if not distress.empty:
            d_min = pd.to_datetime(distress['year_month'].min())
            d_max = pd.to_datetime(distress['year_month'].max())
            ax.axvspan(d_min, d_max, alpha=0.12, color='red', label='Distress period')

    # Crisis event marker
    ax.axvline(crisis_dt, color='red', linestyle='--', linewidth=1.5, alpha=0.8)

    ax.set_title(f"{COUNTRY_NAMES.get(iso3, iso3)} — {crisis_label}", fontsize=13, fontweight='bold')
    ax.set_ylabel('P(Distress)')
    ax.set_ylim(-0.02, 1.02)
    ax.grid(True, alpha=0.2)
    if idx == 0:
        ax.legend(fontsize=9, loc='upper left')

fig.suptitle('Country Risk Timelines — Model Predictions Before Crisis Events',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(PLOT_DIR / 'plot_country_risk_timelines.png', dpi=200, bbox_inches='tight')
plt.show()
print("✓ Plot 1 saved: Country risk timelines for 4 crisis countries")


In [ ]:

# ════════════════════════════════════════════════════════════════
# PLOT 2: Model Comparison — Grouped Bar (AUROC + AUPRC + Recall@20%)
# "Which models perform best across different metrics"
# ════════════════════════════════════════════════════════════════

summary = pd.DataFrame(summary_rows)
hero_summary = summary[summary['model'].isin(HERO_MODELS)].copy()
hero_summary['display'] = hero_summary['model'].map(HERO_LABELS)
hero_summary = hero_summary.sort_values('test__auroc', ascending=False)

fig, ax = plt.subplots(figsize=(14, 7))

metrics = ['test__auroc', 'test__auprc', 'test__recall_at_top20pct']
metric_labels = ['AUROC', 'AUPRC', 'Recall@20%']
metric_colors = ['#3b82f6', '#22c55e', '#f59e0b']

x = np.arange(len(hero_summary))
width = 0.25

for i, (metric, label, color) in enumerate(zip(metrics, metric_labels, metric_colors)):
    vals = hero_summary[metric].values
    bars = ax.bar(x + i * width, vals, width, label=label, color=color, edgecolor='white', linewidth=0.5)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val:.2f}', ha='center', va='bottom', fontsize=8, fontweight='bold')

ax.set_xticks(x + width)
ax.set_xticklabels(hero_summary['display'].values, fontsize=11)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Model Performance Comparison — Test Set', fontsize=15, fontweight='bold')
ax.legend(fontsize=11, loc='upper right')
ax.set_ylim(0, 1.1)
ax.grid(True, axis='y', alpha=0.2)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(PLOT_DIR / 'plot_model_comparison_grouped.png', dpi=200, bbox_inches='tight')
plt.show()
print("✓ Plot 2 saved: Model comparison grouped bar chart")


In [ ]:

# ════════════════════════════════════════════════════════════════
# PLOT 3: ROC Curves — Top Models
# "Discrimination ability across all decision thresholds"
# ════════════════════════════════════════════════════════════════

fig, ax = plt.subplots(figsize=(9, 7))

for model_name in HERO_MODELS:
    if model_name not in test_scores_by_model:
        continue
    scores = test_scores_by_model[model_name]
    # Determine correct y_true
    if model_name in ['logistic_regression']:
        y_true = y_test_flat
    else:
        y_true = y_test_seq

    fpr, tpr, _ = roc_curve(y_true, scores)
    auroc = safe_roc_auc(y_true, scores)
    lw = 3.0 if model_name == 'improved_multimodal_gru_transformer' else 1.8
    ax.plot(fpr, tpr, color=HERO_COLORS[model_name], lw=lw,
            label=f"{HERO_LABELS[model_name]} ({auroc:.3f})")

ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.4, label='Random (0.500)')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — Test Set', fontsize=15, fontweight='bold')
ax.legend(fontsize=10, loc='lower right')
ax.grid(True, alpha=0.2)
ax.set_xlim(-0.01, 1.01)
ax.set_ylim(-0.01, 1.01)

plt.tight_layout()
plt.savefig(PLOT_DIR / 'plot_roc_curves.png', dpi=200, bbox_inches='tight')
plt.show()
print("✓ Plot 3 saved: ROC curves")





In [ ]:
# ════════════════════════════════════════════════════════════════
# PLOT 4: Precision-Recall Curves — Top Models
# "Performance on the rare positive class (most important metric)"
# ════════════════════════════════════════════════════════════════

fig, ax = plt.subplots(figsize=(9, 7))

baseline_rate = y_test_seq.mean()

for model_name in HERO_MODELS:
    if model_name not in test_scores_by_model:
        continue
    scores = test_scores_by_model[model_name]
    y_true = y_test_flat if model_name == 'logistic_regression' else y_test_seq

    prec, rec, _ = precision_recall_curve(y_true, scores)
    auprc = float(average_precision_score(y_true, scores))
    lw = 3.0 if model_name == 'improved_multimodal_gru_transformer' else 1.8
    ax.plot(rec, prec, color=HERO_COLORS[model_name], lw=lw,
            label=f"{HERO_LABELS[model_name]} ({auprc:.3f})")

ax.axhline(baseline_rate, color='gray', ls='--', lw=1, alpha=0.6,
           label=f'No-skill baseline ({baseline_rate:.3f})')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curves — Test Set', fontsize=15, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.2)
ax.set_xlim(-0.01, 1.01)
ax.set_ylim(-0.01, 1.01)

plt.tight_layout()
plt.savefig(PLOT_DIR / 'plot_pr_curves.png', dpi=200, bbox_inches='tight')
plt.show()
print("✓ Plot 4 saved: Precision-recall curves")


In [ ]:

# ════════════════════════════════════════════════════════════════
# PLOT 5: Prediction Heatmap — All Countries × Best Model
# "Which countries does the model flag as high risk?"
# ════════════════════════════════════════════════════════════════

best_model = 'improved_multimodal_gru_transformer'
val_df = load_preds(best_model, 'val')
test_df = load_preds(best_model, 'test')
all_preds = pd.concat([val_df, test_df]).sort_values(['iso3', 'year_month'])
all_preds['date'] = pd.to_datetime(all_preds['year_month'])
all_preds['month_label'] = all_preds['date'].dt.strftime('%Y-%m')

pivot = all_preds.pivot_table(index='iso3', columns='month_label', values='score', aggfunc='first')

# Sort countries: crisis countries on top, stable on bottom
crisis_isos = all_preds[all_preds['y'] == 1]['iso3'].unique()
stable_isos = [c for c in pivot.index if c not in crisis_isos]
sorted_isos = list(crisis_isos) + sorted(stable_isos)
pivot = pivot.reindex(sorted_isos)

# Replace iso3 with country names for display
pivot.index = [COUNTRY_NAMES.get(iso, iso) for iso in pivot.index]

fig, ax = plt.subplots(figsize=(20, 10))
im = ax.imshow(pivot.values, aspect='auto', cmap='RdYlGn_r', vmin=0, vmax=0.8, interpolation='nearest')

ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index, fontsize=9)

# Show every 6th month on x-axis
n_months = len(pivot.columns)
tick_positions = list(range(0, n_months, 6))
ax.set_xticks(tick_positions)
ax.set_xticklabels([pivot.columns[i] for i in tick_positions], fontsize=8, rotation=45, ha='right')

ax.set_title(f'Predicted Distress Probability — {HERO_LABELS[best_model]}',
             fontsize=15, fontweight='bold')

cbar = plt.colorbar(im, ax=ax, shrink=0.7, pad=0.02)
cbar.set_label('P(Distress within 12 months)', fontsize=11)

# Draw horizontal line separating crisis from stable countries
if len(crisis_isos) > 0:
    ax.axhline(len(crisis_isos) - 0.5, color='white', linewidth=2)
    ax.text(n_months + 1, len(crisis_isos) / 2 - 0.5, '← Crisis\n    countries',
            fontsize=9, color='red', va='center', fontweight='bold')
    ax.text(n_months + 1, len(crisis_isos) + len(stable_isos) / 2, '← Stable\n    countries',
            fontsize=9, color='green', va='center', fontweight='bold')

plt.tight_layout()
plt.savefig(PLOT_DIR / 'plot_prediction_heatmap.png', dpi=200, bbox_inches='tight')
plt.show()
print("✓ Plot 5 saved: Country prediction heatmap")








In [ ]:

# ════════════════════════════════════════════════════════════════
# PLOT 6: Stable vs Crisis Countries — Score Distribution
# "The model clearly separates stable countries from crisis countries"
# ════════════════════════════════════════════════════════════════

best_model = 'improved_multimodal_gru_transformer'
test_df = load_preds(best_model, 'test')

crisis_scores = test_df[test_df['y'] == 1]['score'].values
stable_scores = test_df[test_df['y'] == 0]['score'].values

fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(stable_scores, bins=50, alpha=0.7, color='#22c55e', label=f'No distress (n={len(stable_scores)})', density=True)
ax.hist(crisis_scores, bins=20, alpha=0.7, color='#ef4444', label=f'Distress within 12mo (n={len(crisis_scores)})', density=True)

ax.set_xlabel('Predicted Probability', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Score Distribution — Crisis vs Stable Country-Months', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig(PLOT_DIR / 'plot_score_distribution.png', dpi=200, bbox_inches='tight')
plt.show()
print("✓ Plot 6 saved: Score distribution (crisis vs stable)")


In [ ]:
# ════════════════════════════════════════════════════════════════
# PLOT 7: Validation vs Test Performance
# "Showing model generalization — val results predict test performance"
# ════════════════════════════════════════════════════════════════

summary = pd.DataFrame(summary_rows)
hero_summary = summary[summary['model'].isin(HERO_MODELS)].copy()
hero_summary['display'] = hero_summary['model'].map(HERO_LABELS)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# AUROC: val vs test
ax = axes[0]
for _, row in hero_summary.iterrows():
    model = row['model']
    ax.scatter(row['val__auroc'], row['test__auroc'],
               color=HERO_COLORS.get(model, '#999'),
               s=120, zorder=5, edgecolors='white', linewidth=1.5)
    ax.annotate(row['display'], (row['val__auroc'], row['test__auroc']),
                textcoords="offset points", xytext=(8, 5), fontsize=9)
ax.plot([0.4, 1], [0.4, 1], 'k--', alpha=0.3, lw=1)
ax.set_xlabel('Validation AUROC')
ax.set_ylabel('Test AUROC')
ax.set_title('AUROC: Validation vs Test', fontweight='bold')
ax.grid(True, alpha=0.2)
ax.set_xlim(0.55, 0.95)
ax.set_ylim(0.35, 1.0)

# AUPRC: val vs test
ax = axes[1]
for _, row in hero_summary.iterrows():
    model = row['model']
    ax.scatter(row['val__auprc'], row['test__auprc'],
               color=HERO_COLORS.get(model, '#999'),
               s=120, zorder=5, edgecolors='white', linewidth=1.5)
    ax.annotate(row['display'], (row['val__auprc'], row['test__auprc']),
                textcoords="offset points", xytext=(8, 5), fontsize=9)
ax.plot([0, 0.5], [0, 0.5], 'k--', alpha=0.3, lw=1)
ax.set_xlabel('Validation AUPRC')
ax.set_ylabel('Test AUPRC')
ax.set_title('AUPRC: Validation vs Test', fontweight='bold')
ax.grid(True, alpha=0.2)

fig.suptitle('Model Generalization — Validation vs Test Performance',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(PLOT_DIR / 'plot_val_vs_test.png', dpi=200, bbox_inches='tight')
plt.show()
print("✓ Plot 7 saved: Validation vs test generalization")


In [ ]:
# ════════════════════════════════════════════════════════════════
# PLOT 8: Summary Results Table (publication-ready)
# "Clean table for the report and PPT slides"
# ════════════════════════════════════════════════════════════════

summary = pd.DataFrame(summary_rows)
hero_summary = summary[summary['model'].isin(HERO_MODELS)].copy()
hero_summary['Model'] = hero_summary['model'].map(HERO_LABELS)
hero_summary = hero_summary.sort_values('test__auroc', ascending=False)

table_data = hero_summary[[
    'Model', 'test__auroc', 'test__auprc',
    'test__recall_at_top10pct', 'test__recall_at_top20pct'
]].copy()
table_data.columns = ['Model', 'AUROC', 'AUPRC', 'Recall@10%', 'Recall@20%']

fig, ax = plt.subplots(figsize=(12, 4))
ax.axis('off')

# Color cells
cell_colors = []
for _, row in table_data.iterrows():
    row_colors = ['#f8fafc']  # Model name column
    for val in [row['AUROC'], row['AUPRC'], row['Recall@10%'], row['Recall@20%']]:
        if val >= 0.5:
            row_colors.append('#dcfce7')  # Green
        elif val >= 0.2:
            row_colors.append('#fef9c3')  # Yellow
        else:
            row_colors.append('#fee2e2')  # Red
    cell_colors.append(row_colors)

table = ax.table(
    cellText=[[row['Model'], f"{row['AUROC']:.3f}", f"{row['AUPRC']:.3f}",
               f"{row['Recall@10%']:.1%}", f"{row['Recall@20%']:.1%}"]
              for _, row in table_data.iterrows()],
    colLabels=['Model', 'AUROC ↑', 'AUPRC ↑', 'Recall@10% ↑', 'Recall@20% ↑'],
    cellColours=cell_colors,
    colColours=['#e2e8f0'] * 5,
    loc='center',
    cellLoc='center'
)
table.auto_set_font_size(False)
table.set_fontsize(12)
table.scale(1.2, 2.0)

# Bold header
for (r, c), cell in table.get_celld().items():
    if r == 0:
        cell.set_text_props(fontweight='bold')
    cell.set_edgecolor('#cbd5e1')

ax.set_title('Model Performance Summary — Test Set (2022–2024)',
             fontsize=15, fontweight='bold', pad=20)

plt.tight_layout()
plt.savefig(PLOT_DIR / 'plot_results_table.png', dpi=200, bbox_inches='tight')
plt.show()
print("✓ Plot 8 saved: Publication-ready results table")


In [ ]:


# ════════════════════════════════════════════════════════════════
# FINAL SUMMARY
# ════════════════════════════════════════════════════════════════

print("\n" + "=" * 60)
print("ALL VISUALIZATIONS COMPLETE")
print("=" * 60)
print(f"\nSaved to: {PLOT_DIR}/")
print("""
Plots created:
  1. plot_country_risk_timelines.png    — Risk buildup before 4 crises
  2. plot_model_comparison_grouped.png  — AUROC + AUPRC + Recall bar chart
  3. plot_roc_curves.png                — ROC curves for 6 models
  4. plot_pr_curves.png                 — Precision-recall curves
  5. plot_prediction_heatmap.png        — All countries × all months heatmap
  6. plot_score_distribution.png        — Crisis vs stable score separation
  7. plot_val_vs_test.png               — Generalization scatter plot
  8. plot_results_table.png             — Publication-ready comparison table
""")

In [ ]:
# ════════════════════════════════════════════════════════════════
# PLOT 9: Lead-Time Analysis
# "How many months before each crisis did the model raise the alarm?"
# ════════════════════════════════════════════════════════════════

# Curated crisis events in the test period (2022-2024)
CRISIS_EVENTS = [
    {'iso3': 'LKA', 'date': '2022-05-01', 'label': 'Sri Lanka'},
    {'iso3': 'GHA', 'date': '2022-12-01', 'label': 'Ghana'},
    {'iso3': 'ETH', 'date': '2023-12-01', 'label': 'Ethiopia'},
    {'iso3': 'PAK', 'date': '2023-01-01', 'label': 'Pakistan'},
    {'iso3': 'KEN', 'date': '2024-07-01', 'label': 'Kenya'},
]

# Also include val-period events if predictions cover them
VAL_CRISIS_EVENTS = [
    {'iso3': 'ZMB', 'date': '2020-11-01', 'label': 'Zambia'},
    {'iso3': 'LBN', 'date': '2020-03-01', 'label': 'Lebanon'},
    {'iso3': 'ARG', 'date': '2020-05-01', 'label': 'Argentina'},
    {'iso3': 'ECU', 'date': '2020-04-01', 'label': 'Ecuador'},
]

ALL_CRISIS_EVENTS = CRISIS_EVENTS + VAL_CRISIS_EVENTS

models_for_leadtime = {
    'improved_multimodal_gru_transformer': ('Multimodal GRU-TF', '#ef4444'),
    'gru': ('GRU', '#22c55e'),
    'logistic_regression': ('Logistic Reg.', '#64748b'),
}

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# LEFT: Lead-time histogram
ax = axes[0]
for model_name, (display_name, color) in models_for_leadtime.items():
    val_df = load_preds(model_name, 'val')
    test_df = load_preds(model_name, 'test')
    if val_df is None or test_df is None:
        continue
    combined = pd.concat([val_df, test_df]).copy()
    combined['date'] = pd.to_datetime(combined['year_month'])

    # For each model, find the threshold (top 10% of scores)
    threshold = np.quantile(combined['score'], 0.90)

    lead_times = []
    for ev in ALL_CRISIS_EVENTS:
        crisis_dt = pd.to_datetime(ev['date'])
        lookback_start = crisis_dt - pd.DateOffset(months=12)
        mask = ((combined['iso3'] == ev['iso3']) &
                (combined['date'] >= lookback_start) &
                (combined['date'] < crisis_dt) &
                (combined['score'] >= threshold))
        flagged = combined[mask]
        if len(flagged) > 0:
            earliest = flagged['date'].min()
            lead_months = (crisis_dt - earliest).days / 30.44
            lead_times.append(lead_months)

    if lead_times:
        ax.hist(lead_times, bins=range(0, 14), alpha=0.55, color=color, edgecolor='white',
                label=f'{display_name} (detected {len(lead_times)}/{len(ALL_CRISIS_EVENTS)})')

ax.set_xlabel('Lead Time (months before crisis)', fontsize=12)
ax.set_ylabel('Number of events detected', fontsize=12)
ax.set_title('Lead-Time Distribution', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.2)
ax.set_xticks(range(0, 13))

# RIGHT: Per-event lead-time bar chart (best model only)
ax = axes[1]
best_model = 'improved_multimodal_gru_transformer'
val_df = load_preds(best_model, 'val')
test_df = load_preds(best_model, 'test')
combined = pd.concat([val_df, test_df]).copy()
combined['date'] = pd.to_datetime(combined['year_month'])
threshold = np.quantile(combined['score'], 0.90)

event_labels = []
event_leads = []
event_colors = []

for ev in ALL_CRISIS_EVENTS:
    crisis_dt = pd.to_datetime(ev['date'])
    lookback_start = crisis_dt - pd.DateOffset(months=12)
    mask = ((combined['iso3'] == ev['iso3']) &
            (combined['date'] >= lookback_start) &
            (combined['date'] < crisis_dt) &
            (combined['score'] >= threshold))
    flagged = combined[mask]
    event_labels.append(ev['label'])
    if len(flagged) > 0:
        earliest = flagged['date'].min()
        lead = (crisis_dt - earliest).days / 30.44
        event_leads.append(lead)
        event_colors.append('#22c55e' if lead >= 6 else '#f59e0b' if lead >= 3 else '#ef4444')
    else:
        event_leads.append(0)
        event_colors.append('#dc2626')

y_pos = range(len(event_labels))
ax.barh(y_pos, event_leads, color=event_colors, edgecolor='white', height=0.6)
ax.set_yticks(y_pos)
ax.set_yticklabels(event_labels, fontsize=10)
ax.set_xlabel('Months of advance warning', fontsize=12)
ax.set_title('Per-Event Lead Time (Multimodal GRU-TF)', fontsize=14, fontweight='bold')
ax.grid(True, axis='x', alpha=0.2)

# Add value labels
for i, (lead, label) in enumerate(zip(event_leads, event_labels)):
    if lead > 0:
        ax.text(lead + 0.2, i, f'{lead:.0f}mo', va='center', fontsize=10, fontweight='bold')
    else:
        ax.text(0.2, i, 'Not detected', va='center', fontsize=9, color='#dc2626', fontstyle='italic')

# Legend
from matplotlib.patches import Patch
legend_items = [
    Patch(facecolor='#22c55e', label='≥6 months (excellent)'),
    Patch(facecolor='#f59e0b', label='3–6 months (good)'),
    Patch(facecolor='#ef4444', label='<3 months or missed'),
]
ax.legend(handles=legend_items, fontsize=9, loc='lower right')

fig.suptitle('Lead-Time Analysis — How Early Does the Model Detect Crises?',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(PLOT_DIR / 'plot_lead_time_analysis.png', dpi=200, bbox_inches='tight')
plt.show()
print("✓ Plot 9 saved: Lead-time analysis")


In [ ]:

# ════════════════════════════════════════════════════════════════
# PLOT 10: Top-20 Riskiest Predictions — Was the model right?
# "Of the 20 highest-risk alerts, how many were actual crises?"
# ════════════════════════════════════════════════════════════════

best_model = 'improved_multimodal_gru_transformer'
test_df = load_preds(best_model, 'test').copy()
test_df['date'] = pd.to_datetime(test_df['year_month'])
test_df['country'] = test_df['iso3'].map(COUNTRY_NAMES)
test_df['date_str'] = test_df['date'].dt.strftime('%b %Y')

# Top 20 by predicted probability
top20 = test_df.nlargest(20, 'score').reset_index(drop=True)

fig, ax = plt.subplots(figsize=(14, 8))
ax.axis('off')

# Build table data
table_rows = []
row_colors = []
for i, row in top20.iterrows():
    correct = '✓ Crisis' if row['y'] == 1 else '✗ No crisis'
    table_rows.append([
        f"{i+1}",
        row['country'],
        row['date_str'],
        f"{row['score']:.4f}",
        correct
    ])
    if row['y'] == 1:
        row_colors.append(['#f0fdf4', '#f0fdf4', '#f0fdf4', '#dcfce7', '#dcfce7'])
    else:
        row_colors.append(['#fff7ed', '#fff7ed', '#fff7ed', '#fef3c7', '#fee2e2'])

table = ax.table(
    cellText=table_rows,
    colLabels=['Rank', 'Country', 'Date', 'Predicted Risk', 'Actual Outcome'],
    cellColours=row_colors,
    colColours=['#e2e8f0'] * 5,
    loc='center',
    cellLoc='center'
)
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1.1, 1.6)

for (r, c), cell in table.get_celld().items():
    if r == 0:
        cell.set_text_props(fontweight='bold')
    cell.set_edgecolor('#cbd5e1')
    # Color the outcome column text
    if c == 4 and r > 0:
        if '✓' in cell.get_text().get_text():
            cell.get_text().set_color('#16a34a')
            cell.get_text().set_fontweight('bold')
        else:
            cell.get_text().set_color('#dc2626')

# Count hits
hits = top20['y'].sum()
total = len(top20)
ax.set_title(
    f'Top-20 Highest Risk Predictions — {hits}/{total} Correct ({hits/total:.0%} Precision)\n'
    f'Model: Multimodal GRU-Transformer',
    fontsize=15, fontweight='bold', pad=20
)

plt.tight_layout()
plt.savefig(PLOT_DIR / 'plot_top20_predictions.png', dpi=200, bbox_inches='tight')
plt.show()
print(f"✓ Plot 10 saved: Top-20 predictions table ({hits}/{total} correct)")




In [ ]:

# ════════════════════════════════════════════════════════════════
# PLOT 11: Modality Contribution Analysis
# "Which signal types matter most? How much does each add?"
#
# We approximate this using feature importance from tree models
# grouped by modality, since we have the modality map.
# If you ran ablation experiments, replace this with actual ablation AUROC.
# ════════════════════════════════════════════════════════════════

# Get XGBoost feature importances (already trained as xgb_flat)
try:
    importances = xgb_flat.feature_importances_
    feat_names = FLAT_FEATURES

    # Map each feature to its modality
    feat_to_modality = {}
    for mod, feats in MODALITY_MAP.items():
        for f in feats:
            if f in feat_names:
                feat_to_modality[f] = mod

    # Aggregate importance by modality
    modality_importance = {}
    for feat, imp in zip(feat_names, importances):
        mod = feat_to_modality.get(feat, 'other')
        if mod not in modality_importance:
            modality_importance[mod] = 0
        modality_importance[mod] += imp

    # Sort and plot
    mod_df = pd.DataFrame([
        {'Modality': mod.replace('_', ' ').title(), 'Importance': imp}
        for mod, imp in modality_importance.items()
        if imp > 0 and mod != 'other'
    ]).sort_values('Importance', ascending=True)

    modality_colors = {
        'Macro': '#3b82f6',
        'Debt': '#ef4444',
        'Political': '#f59e0b',
        'Bop': '#8b5cf6',
        'Global': '#06b6d4',
        'Structural': '#22c55e',
        'Commodity': '#ec4899',
        'Liquidity': '#14b8a6',
        'Missing Flags': '#94a3b8',
        'Engineered': '#6366f1',
        'Labels': '#64748b',
    }

    fig, axes = plt.subplots(1, 2, figsize=(16, 7))

    # LEFT: Modality importance bar chart
    ax = axes[0]
    colors = [modality_colors.get(m, '#94a3b8') for m in mod_df['Modality']]
    ax.barh(range(len(mod_df)), mod_df['Importance'].values, color=colors, edgecolor='white', height=0.6)
    ax.set_yticks(range(len(mod_df)))
    ax.set_yticklabels(mod_df['Modality'].values, fontsize=11)
    ax.set_xlabel('Aggregated Feature Importance', fontsize=12)
    ax.set_title('Importance by Signal Modality', fontsize=14, fontweight='bold')
    ax.grid(True, axis='x', alpha=0.2)

    # Add percentage labels
    total_imp = mod_df['Importance'].sum()
    for i, (_, row) in enumerate(mod_df.iterrows()):
        pct = row['Importance'] / total_imp * 100
        ax.text(row['Importance'] + total_imp * 0.01, i, f'{pct:.1f}%',
                va='center', fontsize=10, fontweight='bold')

    # RIGHT: Pie chart for modality contribution
    ax = axes[1]
    pie_df = mod_df.sort_values('Importance', ascending=False)
    pie_colors = [modality_colors.get(m, '#94a3b8') for m in pie_df['Modality']]
    wedges, texts, autotexts = ax.pie(
        pie_df['Importance'], labels=pie_df['Modality'],
        colors=pie_colors, autopct='%1.1f%%', startangle=90,
        pctdistance=0.75, textprops={'fontsize': 10}
    )
    for at in autotexts:
        at.set_fontweight('bold')
        at.set_fontsize(9)
    ax.set_title('Modality Contribution Share', fontsize=14, fontweight='bold')

    fig.suptitle('Which Signal Types Drive Predictions?',
                 fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(PLOT_DIR / 'plot_modality_contribution.png', dpi=200, bbox_inches='tight')
    plt.show()
    print("✓ Plot 11 saved: Modality contribution analysis")

except Exception as e:
    print(f"⚠ Plot 11 skipped (XGBoost not available): {e}")
    print("  To create this plot, run after the XGBoost training cell.")




In [ ]:

# ════════════════════════════════════════════════════════════════
# PLOT 12: Top-15 Individual Feature Importance (XGBoost)
# "The specific variables that matter most for predicting crises"
# ════════════════════════════════════════════════════════════════

try:
    importances = xgb_flat.feature_importances_
    feat_names = FLAT_FEATURES

    feat_imp = pd.DataFrame({
        'feature': feat_names,
        'importance': importances
    }).sort_values('importance', ascending=False).head(15)

    # Assign colors by modality
    def get_modality(feat):
        for mod, feats in MODALITY_MAP.items():
            if feat in feats:
                return mod
        return 'other'

    feat_imp['modality'] = feat_imp['feature'].apply(get_modality)

    modality_colors_lower = {
        'macro': '#3b82f6', 'debt': '#ef4444', 'political': '#f59e0b',
        'bop': '#8b5cf6', 'global': '#06b6d4', 'structural': '#22c55e',
        'commodity': '#ec4899', 'liquidity': '#14b8a6', 'other': '#94a3b8',
        'missing_flags': '#94a3b8', 'engineered': '#6366f1', 'labels': '#64748b',
    }

    fig, ax = plt.subplots(figsize=(12, 8))
    feat_imp_plot = feat_imp.iloc[::-1]  # Reverse for horizontal bar
    colors = [modality_colors_lower.get(m, '#94a3b8') for m in feat_imp_plot['modality']]

    ax.barh(range(len(feat_imp_plot)), feat_imp_plot['importance'].values,
            color=colors, edgecolor='white', height=0.65)
    ax.set_yticks(range(len(feat_imp_plot)))

    # Clean up feature names for display
    display_names = []
    for f in feat_imp_plot['feature']:
        name = f.replace('_', ' ').replace('edt ', 'Debt: ').replace('bop ', 'BOP: ')
        name = name.replace('acled ', 'ACLED: ').replace('global ', 'Global: ')
        name = name.replace('weo ', 'WEO: ').replace('fx ', 'FX: ')
        name = name.replace('cpi ', 'CPI: ').replace('ir ', 'Rate: ')
        display_names.append(name[:40])

    ax.set_yticklabels(display_names, fontsize=10)
    ax.set_xlabel('Feature Importance (XGBoost gain)', fontsize=12)
    ax.set_title('Top-15 Most Important Features', fontsize=15, fontweight='bold')
    ax.grid(True, axis='x', alpha=0.2)

    # Legend by modality
    seen = set()
    legend_items = []
    for _, row in feat_imp_plot.iterrows():
        m = row['modality']
        if m not in seen and m != 'other':
            seen.add(m)
            legend_items.append(Patch(facecolor=modality_colors_lower.get(m, '#94a3b8'),
                                     label=m.replace('_', ' ').title()))
    ax.legend(handles=legend_items, fontsize=9, loc='lower right')

    plt.tight_layout()
    plt.savefig(PLOT_DIR / 'plot_top_features.png', dpi=200, bbox_inches='tight')
    plt.show()
    print("✓ Plot 12 saved: Top-15 feature importance")

except Exception as e:
    print(f"⚠ Plot 12 skipped: {e}")



In [ ]:
# ════════════════════════════════════════════════════════════════
# SUMMARY OF ALL HIGH-IMPACT PLOTS
# ════════════════════════════════════════════════════════════════

print("\n" + "=" * 60)
print("HIGH-IMPACT VISUALIZATIONS COMPLETE")
print("=" * 60)
print(f"""
Plots 9-12 saved to: {PLOT_DIR}/

  9.  plot_lead_time_analysis.png
      → Left: Lead-time histogram (3 models compared)
      → Right: Per-event bar chart showing months of warning
      → PPT: "Our model provides 6-12 months advance warning"

  10. plot_top20_predictions.png
      → Table of 20 highest-risk predictions with ✓/✗ outcomes
      → PPT: "X out of 20 top alerts were correct crises"

  11. plot_modality_contribution.png
      → Left: Bar chart of importance by modality
      → Right: Pie chart of modality share
      → PPT: "All signal types contribute — multimodal fusion is justified"

  12. plot_top_features.png
      → Top 15 individual features, color-coded by modality
      → PPT: "Reserve levels, debt ratios, and political instability
              are the strongest predictors"
""")
